# 04. Decision Trees - Iris Classification

## What is this notebook about?

A **Decision Tree** is a flowchart of yes/no questions. It splits the data step by step, asking the most useful question at each node. The leaves of the tree are the predictions.

Decision Trees are:
- **Interpretable** - you can read the tree and explain its decisions
- **Non-linear** - they can capture complex patterns
- **Prone to overfitting** - left unchecked, they memorize the training data

## What you will learn

1. How to **train** a decision tree
2. How to **visualize** the tree as a flowchart
3. How **tree depth** controls overfitting
4. How to read **feature importance**

## Dataset

The classic **Iris** dataset: 150 flowers, 4 measurements (sepal length/width, petal length/width), 3 species.

In [ ]:
# If you're running locally, uncomment and run this once.
# In Google Colab, all of these are pre-installed - you can skip this cell.
# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebook
import numpy as np                 # numerical arrays
import pandas as pd                # tabular data (DataFrames)
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier statistical plots

# Set up plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Set a random seed so results are reproducible
np.random.seed(42)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

# Load Iris (built into sklearn)
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

print(f"Shape: {X.shape}")
print(f"Classes: {iris.target_names}")
X.head()

## Step 1. Train a shallow tree (max_depth=3)

`max_depth=3` means the tree can ask at most 3 questions in a row before making a prediction. Limiting depth prevents overfitting.

In [ ]:
# Train/test split with stratification (keeps class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Build the tree
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

# How does it score on training and test data?
print(f"Train accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.3f}")

## Step 2. Visualize the Tree

This is the killer feature of Decision Trees - you can literally read the model.

In [ ]:
# plot_tree draws the flowchart of decisions
plt.figure(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=X.columns,         # what to label each split
    class_names=iris.target_names,   # what to call each leaf
    filled=True,                     # color leaves by predicted class
    rounded=True,
)
plt.title("Decision Tree (max_depth=3)")
plt.show()

**How to read this tree:**
1. Start at the top node (root).
2. Follow the **left** branch if the condition is **True**, **right** if **False**.
3. Each leaf shows the predicted class.

A doctor or business stakeholder can look at this tree and understand exactly how the model decides.

## Step 3. Watch the tree overfit

Now let's grow the tree deeper and see what happens to training vs test accuracy.

In [ ]:
# Try depths 1 through 15
train_accs = []
test_accs = []
depths = range(1, 16)

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_accs.append(t.score(X_train, y_train))
    test_accs.append(t.score(X_test, y_test))

# Plot the two curves
plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, "o-", label="Training accuracy", color="black")
plt.plot(depths, test_accs,  "o-", label="Test accuracy",     color="gray")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Train vs Test Accuracy by Depth")
plt.legend()
plt.show()

**The classic overfitting picture:**
- **Training accuracy** keeps rising toward 1.0 as the tree grows.
- **Test accuracy** plateaus or even drops past a certain depth.

The sweet spot is somewhere in the middle. For Iris, depth 3-4 is great.

## Step 4. Feature Importance

Decision Trees give you a free score for how useful each feature was.

In [ ]:
# feature_importances_ sums to 1.0 across all features
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values()

# Plot as horizontal bar chart
plt.figure(figsize=(8, 4))
importance.plot.barh(color="gray")
plt.title("Feature Importance")
plt.xlabel("Relative importance (sums to 1.0)")
plt.show()

print(importance)

**Petal length and petal width** dominate. The tree barely uses sepal measurements. This matches biology: petals differ much more between species than sepals.

## Step 5. Exercises

1. **Try `criterion="entropy"`** instead of the default `"gini"`. Does it change the tree?

2. **Set `min_samples_leaf=10`** - prevents leaves with fewer than 10 samples. Compare to depth limit.

3. **Try the Wine dataset** instead of Iris:
   ```python
   from sklearn.datasets import load_wine
   wine = load_wine(as_frame=True)
   ```

4. **Train a `DecisionTreeRegressor`** on California Housing data. The visualization works the same way.

5. **Export the tree as text** with `from sklearn.tree import export_text` and read it.

Next: **05 - Random Forest** (a forest of decision trees)!